# Venturi Control & Disruption Experiments

The **Venturi controller** is a hierarchical hybrid stack:

- **Slow loop (~100 ms):** Bayesian rotational prior sets sweep/RMP bounds
- **Fast loop (~1 ms):** traffic-aware divertor circularization
- **Watchdog:** PID safety override on engineering limit breach

`DisruptionDetector` fuses HELIX SNR, trainable ELM predictor, and MHD stability margins into actuation recommendations.

---

## Hierarchical control as nested optimization (VISION §5)

**Slow loop** (Bayesian prior, ~100 ms): maps global equilibrium features $\mathbf{s} = (\bar{n}_e, \beta_N, f_{rot}, q_{95})$ to policy bounds $(v_{sweep}^{max}, \phi_{RMP}^{max})$.

**Fast loop** (PPO / traffic router, ~1 ms): maps divertor heat flux map $Q(\theta, \phi)$ to actuation $\mathbf{a} = (v_{sweep}, \phi_{RMP}, \dot{n}_{gas})$.

**Watchdog** (hardware): if $Q_{peak} > 0.8\, Q_{eng}$, override $\mathbf{a} \leftarrow \mathbf{a}_{safe}$.

### Circularization reward

$$\mathcal{R} = -\mathrm{Var}_t(Q_{divertor}) - 0.1\, Q_{peak} + \mathbb{1}_{\{\mathrm{Var} < 0.5\}} \cdot 5 - \mathbb{1}_{\{watchdog\}} \cdot 10$$

### Plasma traffic router (novel abstraction)

Congestion ratio $\chi = Q_{peak} / Q_{eng}$. The router treats SOL exhaust like **network load balancing** — when $\chi \to 1$, increase sweep and RMP phase to redistribute flux toroidally (analogy: TCP packets across servers).

### Disruption fusion functional

$$P_{dis} = \mathrm{clip}\big(0.45\, P_{ELM} + 0.35\, P_{MHD} + 0.2\, P_{HELIX},\, 0,\, 1\big)$$

where $P_{MHD}$ derives from $(q_{min}, \beta_N)$ stability margins — **cross-domain fusion of diagnostics, ML, and MHD** unique to fuselk.


In [ ]:
import sys
from pathlib import Path

repo = Path.cwd()
if not (repo / "src" / "deepiri_fuselk").exists() and (repo.parent / "src" / "deepiri_fuselk").exists():
    repo = repo.parent

try:
    import deepiri_fuselk  # noqa: F401
except ImportError:
    sys.path.insert(0, str(repo / "src"))
    import deepiri_fuselk  # noqa: F401

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
%config InlineBackend.figure_formats = ["retina"]


In [ ]:
from deepiri_fuselk.control.venturi_controller import VenturiController
from deepiri_fuselk.helix import HelixEngine
from deepiri_fuselk.models.disruption_detector import DisruptionDetector
from deepiri_fuselk.models.elm_predictor import ELMPredictor
from deepiri_fuselk.sim import generate_ece_shot, generate_corpus

venturi = VenturiController(engineering_limit=10.0)
prior = venturi.slow_loop(ne_pedestal=0.9, beta_n=2.8, rotation_khz=5.5, q95=3.2)

# 2D divertor heat-flux map (hotspot)
x = np.linspace(-1, 1, 32)
X, Y = np.meshgrid(x, x)
heat_flux = 8.0 * np.exp(-(X**2 + Y**2) / 0.3)
action = venturi.fast_loop(heat_flux, prior, elm_probability=0.75)
state = venturi.step(heat_flux, elm_probability=0.75)

print(f"Prior max sweep: {prior.max_sweep_velocity:.2f}, max RMP: {prior.max_rmp_phase:.2f}")
print(f"Action: sweep={action.sweep_velocity:.2f}, RMP={action.rmp_phase:.2f}, gas={action.gas_puff:.1f}")
print(f"Pellet ready: {action.pellet_ready}, watchdog override: {action.overridden}")
print(f"Venturi reward: {state.reward:.3f}")


In [ ]:
# Train ELM predictor on synthetic corpus
corpus = generate_corpus(n_shots=200, grid_size=24, seed=11)
elm = ELMPredictor()
acc = elm.train_from_corpus(corpus)
detector = DisruptionDetector(elm)

shot = generate_ece_shot(32, seed=99, island_amplitude=0.85)
helix = HelixEngine().process(shot.heat_field, shot.raw_signal, shot.angles)
assessment = detector.assess(helix, q_min=1.8, beta_n=3.2)

print(f"ELM train accuracy: {acc:.1%}")
print(f"Disruption probability: {assessment.probability:.2f}")
print(f"Recommended action: {assessment.recommended_action}")
print(f"Time to disruption: {assessment.time_to_disruption_ms:.1f} ms")


## Experiment: heat-flux profile sweep

In [ ]:
profiles = []
rewards = []
x = np.linspace(-1, 1, 32)
X, Y = np.meshgrid(x, x)
for peak in np.linspace(2, 12, 11):
    hf = peak * np.exp(-(X**2 + Y**2) / 0.4)
    st = venturi.step(hf, elm_probability=0.3)
    profiles.append(peak)
    rewards.append(st.reward)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(profiles, rewards, "o-", color="C6")
ax.axvline(10.0, color="red", ls="--", alpha=0.6, label="engineering limit")
ax.set_xlabel("peak heat flux [MW/m² proxy]"); ax.set_ylabel("Venturi reward")
ax.set_title("Divertor circularization vs. peak load")
ax.legend()
plt.tight_layout()
plt.show()


## Optional: PPO vent training (slow — ~30 s)

Uncomment to train a strike-point sweep policy with stable-baselines3.


In [ ]:
# from deepiri_fuselk.control.rl_agent import train_vent_policy
# trained = train_vent_policy(timesteps=3000)
# print(f"Mean reward: {trained.mean_reward:.3f}, policy: {trained.policy_path}")
print("RL training cell skipped (uncomment to run)")
